# ETL Design - Transform Phase

## Data Transformations

Before loading data into the data warehouse, several transformations are applied:

1. Joining related tables
   - Customer data is joined with address, city, and country tables.
   - Film data is joined with category and language tables.

2. Data Cleaning
   - Remove duplicates.
   - Handle missing values.
   - Standardize text fields such as city, country, and language names.

3. Full Name Creation
   - Combine first_name and last_name into full_name for customers and staff.

4. Surrogate Keys
   - Replace operational IDs with warehouse surrogate keys for dimensions.

5. Date Keys
   - Create date keys from:
     - rental_date
     - return_date
     - payment_date

6. Calculated Measures
   - rental_duration
   - payment_amount
   - rental_count
   - payment_count

7. Late Return Detection
   - Compare actual rental duration with expected rental duration.
   - Mark late rentals using late_return_flag.

8. Many-to-Many Handling
   - Resolve film-category relationships using film_category.
   - Resolve film-actor relationships using film_actor.

9. Historical Change Preparation
   - Prepare dimensions for future handling of attribute changes.

In [2]:
import pymysql
import pandas as pd

conn = pymysql.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="root123",
    database="sakila",
    connect_timeout=5
)
cursor = conn.cursor()

# Extract rental data
cursor.execute("""
    SELECT r.rental_id, r.rental_date, r.return_date,
           r.customer_id, r.staff_id, i.film_id, i.store_id,
           f.rental_duration as allowed_days
    FROM rental r
    JOIN inventory i ON r.inventory_id = i.inventory_id
    JOIN film f ON i.film_id = f.film_id
""")
rows = cursor.fetchall()
df = pd.DataFrame(rows, columns=['rental_id','rental_date','return_date',
                                  'customer_id','staff_id','film_id',
                                  'store_id','allowed_days'])

# Handle missing return_date
df['return_date'] = pd.to_datetime(df['return_date']).fillna(pd.Timestamp.now())
df['rental_date'] = pd.to_datetime(df['rental_date'])

# Calculate rental_duration_days
df['rental_duration_days'] = (df['return_date'] - df['rental_date']).dt.days

# Detect late returns
df['is_late'] = (df['rental_duration_days'] > df['allowed_days']).astype(int)
df['late_days'] = (df['rental_duration_days'] - df['allowed_days']).clip(lower=0)

# Create date_key
df['date_key'] = pd.to_datetime(df['rental_date']).dt.strftime('%Y%m%d').astype(int)

print("Transform Complete ✅")
print(f"Total Rentals: {len(df):,}")
print(f"Late Returns: {df['is_late'].sum():,} ({df['is_late'].mean()*100:.1f}%)")
print(df[['rental_id','rental_duration_days','is_late','late_days','date_key']].head(3))

Transform Complete ✅
Total Rentals: 16,044
Late Returns: 6,586 (41.0%)
   rental_id  rental_duration_days  is_late  late_days  date_key
0       4863                     3        0          0  20050708
1      11433                     9        1          3  20050802
2      14714                     9        1          3  20050821
